# Short-range land surface (`ldasout`)

A **live** pull of one recent `short_range` `land` file — the gridded
Noah-MP land-surface output on the 1 km CONUS grid (~30 MB): soil
moisture, snow water equivalent, snow depth, accumulated ET. Unlike
`chrtout`, `ldasout` is a `raster` product.

## Setup

Imports for the National Water Model pull: `EarthLens` is the unified
entry point, and `tempfile` / `Path` give us a scratch directory for the
downloaded NetCDF.

In [ ]:
import datetime as dt
import tempfile
from pathlib import Path

from earthlens.core import EarthLens

## Pick a cycle and a scratch directory

We target a model cycle from three days ago (recent enough to still be in
the bucket) and write the file into a fresh temporary directory.

In [ ]:
cycle_date = (dt.datetime.now() - dt.timedelta(days=3)).strftime('%Y-%m-%d')
out_dir = Path(tempfile.mkdtemp(prefix='nwm_land_'))

## Build the NWM request

Configure a `short_range` `ldasout` request for two land variables
(`SOIL_M`, `SNEQV`) at cycle `0`, forecast step `1`, over the full grid.
The constructor only describes the request — nothing is fetched yet.

In [ ]:
nwm = EarthLens(
    data_source="nwm",
    start=cycle_date,
    end=cycle_date,
    dataset='ldasout',
    variables=['SOIL_M', 'SNEQV'],
    aoi=[-180, -90, 180, 90],
    configuration='short_range',
    cycles=[0],
    steps=[1],
    path=str(out_dir),
)
print('OUTPUT_KIND:', nwm.OUTPUT_KIND)  # raster

## Download the file

`download()` fetches the gridded NetCDF to `out_dir` and returns the list
of written paths; we print each name and its size.

In [ ]:
paths = nwm.download(progress_bar=False)
for p in paths:
    print(p.name, f'{p.stat().st_size / 1e6:.1f} MB')

The file is the native gridded NetCDF. Cropping it to a bounding box is
a read (pyramids `PY-G` / `pyramids.netcdf.NetCDF`), so the MVP fetches
the whole CONUS grid.

In [ ]:
assert paths, 'expected one land file for a recent short_range cycle'